In [1]:
import os
# import string
import numpy as np
import pandas as pd
# import networkx as nx
from pathlib import Path
from tqdm import tqdm, trange
# import re
# import json
# import pickle

import torch
import torchmetrics
import lightning.pytorch as pl
# from torch.utils.data import Dataset, random_split, DataLoader

from src.utils.misc_utils import load_sharded_dataset, print_time_slices
from src.utils.plotting_utils import plot_lagged_adjacency_structure

import warnings
warnings.filterwarnings("ignore")

roc = torchmetrics.classification.BinaryROC()
auroc = torchmetrics.classification.BinaryAUROC()

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

### Data Permutations

##### Permute train & val splits

In [ ]:
from src.utils.misc_utils import load_sharded_dataset, print_time_slices
# from scipy.stats import ks_2samp
# import matplotlib.pyplot as plt
# import seaborn as sns
# import string
import random

""" Path """
cpd_path = Path("data/mix_100k_pt")
split = "train"
destination_path = Path("data/mix_100k_p3_pt")

""" Data """
data_iterator = load_sharded_dataset(base_path=cpd_path, split=split)

# for j, data in enumerate(data_iterator): 
#     print(len(data))

data = next(data_iterator)
print(len(data))

In [ ]:
from src.utils.misc_utils import load_sharded_dataset, print_time_slices
# from scipy.stats import ks_2samp
# import matplotlib.pyplot as plt
# import seaborn as sns
# import string
import random

""" Path """
cpd_path = Path("data/mix_100k_pt")
split = "test"
destination_path = Path("data/mix_100k_p3_pt")

""" Data """
data_iterator = load_sharded_dataset(base_path=cpd_path, split=split)

for j, data in enumerate(data_iterator): 

    # hyperparameters
    num_permutations = 3

    permuted_data = []
    for pair in [random.choice(data) for _ in range(int(len(data)/4))]:
        permuted_data.append(pair)
        datum = pair[0]
        graph = pair[1]
        datum = datum[:500, :] # trimming the simulated data
        graph = torch.Tensor(graph) # trimming the simulated data
        padded_feats = []
        # sanity checks
        if datum.shape!=torch.Size([500, 12]):
            print(f"Warning: datum.shape={datum.shape}")
        if graph.shape!=torch.Size([12, 12, 3]):
            print(f"Warning: graph.shape={graph.shape}")
        if type(graph)!=torch.Tensor:
            print(f"Warning: graph.type={type(graph)}")
        # identify padded features based on mean (padded ones follow a distribution with a singinficantly smaller mean)
        min_mean = np.min([np.mean(datum.numpy()[:, i]) for i in range(datum.shape[1])])
        mean_interval = pd.Interval(left=min_mean-np.abs(min_mean/2), right=min_mean+np.abs(min_mean/2))
        feats = np.arange(datum.shape[1])
        for i in feats:
            if np.mean(datum.numpy()[:, i]) in mean_interval:
                padded_feats.append(i)
        # permute features at random for both data & graph (assuming the identification of padded features works correctly - seems so)
        for _ in range(num_permutations):
            if padded_feats!=[]:
                permuted_feats = np.concat([
                    np.random.permutation(feats[:min(padded_feats)]), 
                    feats[min(padded_feats):]
                ])
            else:
                permuted_feats = np.random.permutation(feats)
            permuted_datum = datum.clone()[:, permuted_feats]
            permuted_graph = graph.clone()[permuted_feats, :, :][:, permuted_feats, :]
            permuted_data.append([permuted_datum, permuted_graph])

    np.random.shuffle(permuted_data)
    destination_folder = destination_path / split
    os.makedirs(destination_folder, exist_ok=True)
    torch.save(permuted_data, destination_folder / f"{split}_shard{j}.pt")

##### Permute test split into separate groups for each pair (to measure variability of predictions)

In [ ]:
""" Path """
cpd_path = Path("data/mix_100k_pt")
split = "test"
destination_path = Path("data/mix_400k_p3_pt")

destination_split = "testvar"
destination_folder = destination_path / destination_split
os.makedirs(destination_folder, exist_ok=True)

""" Data """
data_iterator = load_sharded_dataset(base_path=cpd_path, split=split)

for j, data in enumerate(data_iterator): 

    # hyperparameters
    num_permutations = 9

    for k, pair in enumerate(data[:]):
        permuted_data = []
        datum = pair[0]
        graph = pair[1]
        datum = datum[:500, :] # trimming the simulated data
        graph = torch.Tensor(graph) # convert all graph types to torch tensor
        permuted_data.append([datum, graph])
        padded_feats = []
        # sanity checks
        if datum.shape!=torch.Size([500, 12]):
            print(f"Warning: datum.shape={datum.shape}")
        if graph.shape!=torch.Size([12, 12, 3]):
            print(f"Warning: graph.shape={graph.shape}")
        if type(graph)!=torch.Tensor:
            print(f"Warning: graph.type={type(graph)}")
        # identify padded features based on mean (padded ones follow a distribution with a singinficantly smaller mean)
        min_mean = np.min([np.mean(datum.numpy()[:, i]) for i in range(datum.shape[1])])
        mean_interval = pd.Interval(left=min_mean-np.abs(min_mean/2), right=min_mean+np.abs(min_mean/2))
        feats = np.arange(datum.shape[1])
        for i in feats:
            if np.mean(datum.numpy()[:, i]) in mean_interval:
                padded_feats.append(i)
        # permute features at random for both data & graph (assuming the identification of padded features works correctly - seems so)
        for _ in range(num_permutations):
            if padded_feats!=[]:
                permuted_feats = np.concat([
                    np.random.permutation(feats[:min(padded_feats)]), 
                    feats[min(padded_feats):]
                ])
            else:
                permuted_feats = np.random.permutation(feats)
            permuted_datum = datum.clone()[:, permuted_feats]
            permuted_graph = graph.clone()[permuted_feats, :, :][:, permuted_feats, :]
            permuted_data.append([permuted_datum, permuted_graph])

        torch.save(permuted_data, destination_folder / f"{destination_split}_shard{j}_{k}.pt")

### Models

In [2]:
from src.modules.informer_module import InformerModule

# Paths
par_dir = Path(os.getcwd())
out_path = Path(par_dir / "outputs")

# try:
#     out_path.mkdir(parents=True, exist_ok=False)
#     print(f"Created: {out_path}")
# except FileExistsError:
#     print(f"{out_path} already exists.")


models = {
    "LCM_CI_plain_100k_2c5M_20E_01_27": InformerModule.load_from_checkpoint(checkpoint_path=Path("logs/plain_100k_2c5M_20E_01_27/last.ckpt")), 
    "LCM_CI_perm_100k_2c5M_20E_01_28": InformerModule.load_from_checkpoint(checkpoint_path=Path("logs/perm_100k_2c5M_20E_01_28/last.ckpt")),
    "LCM_CI_perm_100k_2c5M_20E_01_31": InformerModule.load_from_checkpoint(checkpoint_path=Path("logs/perm_100k_2c5M_20E_01_31/last.ckpt")),
    "LCM_CI_perm_400k_2c5M_3E_01_29": InformerModule.load_from_checkpoint(checkpoint_path=Path("logs/perm_400k_2c5M_3E_01_29/last.ckpt")),
}

### Test Inference

In [ ]:
from src.utils.utils import lagged_batch_crosscorrelation
from src.utils.misc_utils import print_time_slices

data = pd.DataFrame(np.random.rand(500, 12))

X_cpd = torch.tensor(data.values, device='cuda', dtype=torch.float32)

model_name, model = list(models.items())[-1]

model.eval()
if (X_cpd.shape[0]>500):
    X_cpd = X_cpd[:500]
pred = torch.sigmoid(model((X_cpd.unsqueeze(0), lagged_batch_crosscorrelation(X_cpd.unsqueeze(0), 3)))[0])

pred[pred<=0.05] = 0
pred[pred>0.05] = 1

print_time_slices(pred)

In [ ]:
import string
from src.utils.utils import lagged_batch_crosscorrelation, _from_cp_to_full
from src.utils.transformation_utils import _edges_for_causal_stationarity, regular_order_pd, group_lagged_nodes


def estimate_with_LCM(
        true_data: pd.DataFrame,
        model: str,
        thresholded: bool=True,
        threshold: float=0.05,
        enforce_density: bool=False,
        density: list=[2, 10]
) -> tuple: 
    """
    Wrapper for calling LCM models for causal discovery.
    Models used are extended to a maximum of 12 variables a 3 lags, trained on a mixture of synthetic and realistic data. 

    Args
    ----
    data_pd (pd.DataFrame) : a dataframe containing the true time-series data
    model (str) : the path to the PyTorch Lightning model checkpoint
                            trained for up 12 variables and 3 lags, using Correlation Injection (CI) 
    thresholded (bool) : whether to threshold the predicted values or not, in order to have a binary output (default : `True`)
    threshold (floa) : the threshold value used, if thresholded is true (default : `0.05`)
    
    Returns
    ------- 
    (adj_cp, adj_pd) (tuple) : the lagged adjacency matrix (torch.Tensor), together with the full-time graph representation (pandas.DataFrame)

    Notes
    -----
    ...
    """
    true_data.rename(columns=dict(zip(data.columns, list(string.ascii_uppercase)[:len(list(data.columns))])), inplace=True)

    if isinstance(density, list):
        density = np.random.choice(range(density[0], density[1]))

    # Model preparation
    M = InformerModule.load_from_checkpoint(Path(model))
    M = M.to("cpu")
    M = M.eval()
    
    # Data convertion
    data_pd = true_data.copy()
    X_data = torch.tensor(data_pd.values, device='cpu', dtype=torch.float32)

    # Normalization
    X_data = (X_data - X_data.min()) / (X_data.max() - X_data.min())

    # Padding
    MAX_VAR = 12
    VAR_DIF = MAX_VAR - X_data.shape[1]
    if X_data.shape[1] != MAX_VAR:
        X_data = torch.concat(
            [X_data, torch.normal(0, 0.01, (X_data.shape[0], VAR_DIF))], axis=1
        )

    # Check dimensions and decide whether batched approach is needed
    if (X_data.shape[0]>500):
        # bs_preds = []
        # batches = [X_data[500*icr: 500*(icr+1), :] for icr in range(X_data.shape[0]//500)]
        # if 500*(X_data.shape[0]//500) < X_data.shape[0]:
        #     batches.append(X_data[500*(X_data.shape[0]//500):, :])

        # bs_preds = [torch.sigmoid(model((bs.unsqueeze(0), lagged_batch_crosscorrelation(bs.unsqueeze(0), 3)))[0]) for bs in batches]
        # preds = torch.cat(bs_preds, dim=0)
        # pred = preds.mean(0).unsqueeze(0)
        X_data = X_data[:500]
        pred = torch.sigmoid(M((X_data.unsqueeze(0), lagged_batch_crosscorrelation(X_data.unsqueeze(0), 3)))[0])

    else:
        pred = torch.sigmoid(M((X_data.unsqueeze(0), lagged_batch_crosscorrelation(X_data.unsqueeze(0), 3)))[0])

    # Threshold values if a binary output is required
    if thresholded:

        if enforce_density:
            threshold_search_space = np.linspace(5e-7, 0.9, 50)
            dist_to_avg = {}
            for thr in threshold_search_space:
                pred_c = pred[0].detach().numpy().copy()
                pred_c[pred_c < thr] = 0
                pred_c[pred_c >= thr] = 1
                dist_to_avg[thr] = abs(density - pred_c.sum().astype(int))

            dist_to_avg = {k: v for k, v in sorted(dist_to_avg.items(), key=lambda item: item[1])}
            threshold = list(dist_to_avg.keys())[0]

        pred[pred < threshold] = 0
        pred[pred >= threshold] = 1

    adj_cp = pred.detach().numpy()
    # print_time_slices(adj_cp)
    adj_pd = _from_cp_to_full(adj_cp=adj_cp)
    adj_pd = _edges_for_causal_stationarity(temp_adj_pd=adj_pd)

    # __________ Post-processing __________ 
    adj_pd = adj_pd.loc[regular_order_pd(adj_pd=adj_pd), regular_order_pd(adj_pd=adj_pd)]
    adj_pd = adj_pd.loc[
        [col for col in adj_pd.columns if col.split("_t")[0] in data_pd.columns], 
        [col for col in adj_pd.columns if col.split("_t")[0] in data_pd.columns]
    ]
    nodes_to_retain = adj_pd.columns.to_list()
    groups = group_lagged_nodes(adj_pd.columns)
    for key in list(reversed([x for x in groups.keys() if x!="0"])):
        if adj_pd.loc[groups[key], groups['0']].values.sum()==0:
            nodes_to_retain = [col for col in nodes_to_retain if col not in groups[key]]
        else:
            break   # to avoid having empty intermediate time slices
    adj_pd = adj_pd.loc[nodes_to_retain, nodes_to_retain]
    # adj_cp = _from_full_to_cp(adj_pd)

    return adj_cp, adj_pd

data = pd.DataFrame(np.random.rand(500, 5))

model_path = Path("logs/perm_100k_2c5M_20E_01_31/last.ckpt")
adj_cp, adj_pd = estimate_with_LCM(true_data=data, model=model_path)

adj_pd

,A_t,B_t,C_t,D_t,E_t,F_t,G_t,H_t,I_t,J_t,K_t,L_t,A_t-1,B_t-1,C_t-1,D_t-1,E_t-1,F_t-1,G_t-1,H_t-1,I_t-1,J_t-1,K_t-1,L_t-1,A_t-2,B_t-2,C_t-2,D_t-2,E_t-2,F_t-2,G_t-2,H_t-2,I_t-2,J_t-2,K_t-2,L_t-2,A_t-3,B_t-3,C_t-3,D_t-3,E_t-3,F_t-3,G_t-3,H_t-3,I_t-3,J_t-3,K_t-3,L_t-3
A_t,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
B_t,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
C_t,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
D_t,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
E_t,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
F_t,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
G_t,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
H_t,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
I_t,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
J_t,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### Test Sharded

In [ ]:
from src.utils.misc_utils import load_sharded_dataset, print_time_slices
from src.utils.utils import lagged_batch_crosscorrelation
from src.utils.metrics import custom_binary_metrics

""" Path """
cpd_path = Path("data/synth_230k_sim_45k_pt/test")

""" Placeholders """
results_df = pd.DataFrame(columns=["model", "AUC", "TPR", "FPR", "TNR", "FNR"])
results_dict = {}


""" Data """
data_iterator = load_sharded_dataset(base_path=Path("data/synth_230k_sim_45k_pt"), split="test")
data = next(data_iterator)

""" Model loop """
for model_name, model in zip(models.keys(), models.values()):

    print(f"\n___{model_name}___")
    model.eval()

    prc_list = []
    tpr_list = [] 
    fpr_list = []
    tnr_list = []
    fnr_list = []
    auc_list = []
    
    for pair in tqdm(data[:]):

        X_cpd = pd.DataFrame(data=pair[0]) 
        Y_cpd = pair[1] 

#         if "PCMCI" in model_name: 
#             pred = run_inv_pcmci(X_cpd, c_test=None, max_tau=Y_cpd.shape[2], invert=False)

#         elif model_name == "PCMCI+":
#             pred = run_inv_pcmciplus(X_cpd, c_test=None, max_tau=Y_cpd.shape[2], invert=False)
        
#         elif "DYNOTEARS" in model_name:
#             pred_pd = run_dynotears(data=X_cpd, n_lags=Y_cpd.shape[2])
#             pred = _from_full_to_cp(pred_pd)
        
        if "_deep_" or "_lcm_" or "LCM_" in model_name:
            X_cpd = torch.tensor(X_cpd.values, device='cuda', dtype=torch.float32)

            if (X_cpd.shape[0]>500):
                X_cpd = X_cpd[:500]
                if ("LCM" in model_name):
                    pred = torch.sigmoid(model((X_cpd.unsqueeze(0), lagged_batch_crosscorrelation(X_cpd.unsqueeze(0), 3)))[0])
                else:
                    pred = torch.sigmoid(model((X_cpd.unsqueeze(0), lagged_batch_crosscorrelation(X_cpd.unsqueeze(0), 3))))
                pred = pred.unsqueeze(0)

            else:
                if ("LCM" in model_name):    
                    with torch.no_grad():
                        pred = torch.sigmoid(model((X_cpd.unsqueeze(0), lagged_batch_crosscorrelation(X_cpd.unsqueeze(0), 3))))
                else:
                    with torch.no_grad():
                        pred = torch.sigmoid(model(X_cpd.unsqueeze(0)))
            pred = pred[0]


        else:
            continue

        if Y_cpd.sum()<=0 or Y_cpd.sum()==np.prod(Y_cpd.shape):
            continue  

        # prc, tpr, fpr, tnr, fnr, auc = custom_binary_metrics(binary=pred.clone(), A=Y_cpd.clone(), verbose=False, precision=True)
        tpr, fpr, tnr, fnr, auc = custom_binary_metrics(binary=pred.clone(), A=Y_cpd, verbose=False)

        auc_list.append(float(auc))
        # prc_list.append(float(prc))
        tpr_list.append(float(tpr)) 
        fpr_list.append(float(fpr))
        tnr_list.append(float(tnr))
        fnr_list.append(float(fnr))

    results_dict[model_name] = [token for token in auc_list]
    results_df.loc[len(results_df), :] = [model_name, np.array(auc_list).mean(), np.array(tpr_list).mean(), np.array(fpr_list).mean(), 
                        np.array(tnr_list).mean(), np.array(fnr_list).mean()]

results_df[['model', 'AUC', 'TPR', 'FPR', 'TNR', 'FNR']]

### Test Variance

In [10]:
from src.utils.misc_utils import load_sharded_dataset, print_time_slices
from src.utils.utils import lagged_batch_crosscorrelation
from src.utils.metrics import custom_binary_metrics, SHD

""" Placeholders """
total_var_df = pd.DataFrame(
    columns=["auc_var", "shd_var"], 
    index=[model_name for model_name, _ in list(zip(models.keys(), models.values()))[:]],
    data=np.zeros(shape=[len(list(zip(models.keys(), models.values()))[:]), 2])
)

""" Data """
data_iterator = load_sharded_dataset(base_path=Path("data/mix_400k_p3_pt"), split="testvar")
dum_list = []

NUM_GROUPS = 300
SUB = 0
for _ in trange(NUM_GROUPS):
    data = next(data_iterator)

    """ Placeholders """
    var_df = pd.DataFrame(
        columns=["auc_var", "shd_var"], 
        index=[model_name for model_name, _ in list(zip(models.keys(), models.values()))[:]],
    )

    """ Model loop """
    # for model_name, model in zip(models.keys(), models.values()):
    for model_name, model in list(zip(models.keys(), models.values()))[:]:

        # print(f"\n___{model_name}___")
        model.eval()

        tpr_list = [] 
        fpr_list = []
        tnr_list = []
        fnr_list = []
        auc_list = []
        shd_list = []
        
        for pair in tqdm(data[:], disable=True):

            X_cpd = pd.DataFrame(data=pair[0]) 
            Y_cpd = torch.Tensor(pair[1]) 
            
            if "_deep_" or "_lcm_" or "LCM_" in model_name:
                X_cpd = torch.tensor(X_cpd.values, device='cuda', dtype=torch.float32)

                if (X_cpd.shape[0]>500):
                    X_cpd = X_cpd[:500]
                    if ("LCM" in model_name):
                        pred = torch.sigmoid(model((X_cpd.unsqueeze(0), lagged_batch_crosscorrelation(X_cpd.unsqueeze(0), 3)))[0])
                    else:
                        pred = torch.sigmoid(model((X_cpd.unsqueeze(0), lagged_batch_crosscorrelation(X_cpd.unsqueeze(0), 3))))
                    pred = pred.unsqueeze(0)

                else:
                    if ("LCM" in model_name):    
                        with torch.no_grad():
                            pred = torch.sigmoid(model((X_cpd.unsqueeze(0), lagged_batch_crosscorrelation(X_cpd.unsqueeze(0), 3))))
                    else:
                        with torch.no_grad():
                            pred = torch.sigmoid(model(X_cpd.unsqueeze(0)))
                pred = pred[0]


            else:
                continue

            if Y_cpd.sum()<=0 or Y_cpd.sum()==np.prod(Y_cpd.shape):
                continue  

            tpr, fpr, tnr, fnr, auc = custom_binary_metrics(binary=pred.clone(), A=Y_cpd, verbose=False)
            
            
            """ 
            - SHD requires integer adjacency matrices to be computed. 
            - Thresholding the predicted adjacency matrix can be tricky. 
            - Idea: use a threshold such that the number of predicted edges is the same as in the ground truth 
            """
            Y_cpd[Y_cpd < 0.05] = 0
            Y_cpd[Y_cpd >= 0.05] = 1
            density = Y_cpd.sum().item()
            threshold_search_space = np.linspace(5e-7, 0.9, 50)
            dist_to_avg = {}
            for thr in threshold_search_space:
                pred_c = pred.cpu().numpy().copy()
                pred_c[pred_c < thr] = 0
                pred_c[pred_c >= thr] = 1
                dist_to_avg[thr] = abs(density - pred_c.sum().astype(int))

            dist_to_avg = {k: v for k, v in sorted(dist_to_avg.items(), key=lambda item: item[1])}
            threshold = list(dist_to_avg.keys())[0]

            pred[pred < threshold] = 0
            pred[pred >= threshold] = 1

            ssd = SHD(target=Y_cpd, pred=pred)
            # print(f"    - SHD : {ssd}")

            auc_list.append(float(auc))
            tpr_list.append(float(tpr)) 
            fpr_list.append(float(fpr))
            tnr_list.append(float(tnr))
            fnr_list.append(float(fnr))
            shd_list.append(float(ssd))

        if auc_list==[]:
            continue
        var_df.loc[model_name, "auc_var"] = np.array(auc_list).var().round(4)
        var_df.loc[model_name, "shd_var"] = np.array(shd_list).var().round(4)

    # display(var_df)
    for col in total_var_df.columns:
        for ind in total_var_df.index:
            if var_df.loc[ind, col] is np.nan:
                SUB += 1
                break
            else:
                total_var_df.loc[ind, col] = total_var_df.loc[ind, col] + var_df.loc[ind, col]

tvd = (total_var_df / (NUM_GROUPS-SUB)).round(4)
display(tvd)

  0%|          | 0/300 [00:00<?, ?it/s]

- Found 2000 shard(s) for split 'testvar'
  ├── Loaded testvar_shard0_0.pt                 (10 datasamples)


  0%|          | 1/300 [00:00<02:16,  2.19it/s]

  ├── Loaded testvar_shard0_1.pt                 (10 datasamples)


  1%|          | 2/300 [00:00<02:20,  2.12it/s]

...

  ├── Loaded testvar_shard0_367.pt               (10 datasamples)


100%|█████████▉| 299/300 [04:36<00:01,  1.11s/it]

  ├── Loaded testvar_shard0_368.pt               (10 datasamples)


100%|██████████| 300/300 [04:37<00:00,  1.08it/s]


,auc_var,shd_var
LCM_CI_plain_100k_2c5M_20E_01_27,0.0012,5.7100
LCM_CI_perm_100k_2c5M_20E_01_28,0.0020,5.3927
LCM_CI_perm_100k_2c5M_20E_01_31,0.0041,5.9212
LCM_CI_perm_400k_2c5M_3E_01_29,0.0048,9.2913


In [ ]:
tvd

,auc_var,shd_var
LCM_CI_plain_100k_2c5M_20E_01_27,0.0012,5.7100
LCM_CI_perm_100k_2c5M_20E_01_28,0.0020,5.3927
LCM_CI_perm_100k_2c5M_20E_01_31,0.0041,5.9212
LCM_CI_perm_400k_2c5M_3E_01_29,0.0048,9.2913


: 